# LLMBackend Usage

`LLMBackend` implements krrood's `GenerativeBackend` — swapping it onto
`context.query_backend` changes how a `Match` expression gets resolved.

Four patterns:
1. **`resolve_match()`** — Role 2 entry point; hides all krrood wiring
2. **Per-plan wiring** — set backend manually before each `execute_single`
3. **Context-level** — set once at construction; all plans use LLM
4. **Mid-session swap** — mix `ProbabilisticBackend` and `LLMBackend`


In [ ]:
from uniworld import load_pr2_apartment_world
from semantic_digital_twin.world_description.world_entity import Body

world, pr2, context = load_pr2_apartment_world()
groundable_type = Body  # optional — defaults to Symbol inside llm_reasoner


In [ ]:
import os
os.environ['OPENAI_API_KEY'] = ""
from llm_reasoner.reasoning.llm_config import make_llm, LLMProvider

llm = make_llm(LLMProvider.OPENAI, model="gpt-4o-mini")

---
## 1 — `resolve_match()` — Role 2 entry point

The cleanest way to use `LLMBackend` as a pure parameter resolver.
No `context.query_backend` wiring, no `execute_single` import — krrood plumbing
is hidden inside `resolve_match()`.  `instruction` is optional.


In [ ]:
from krrood.entity_query_language.factories import underspecified
from pycram.robot_plans.actions.core.pick_up import PickUpAction

from llm_reasoner import resolve_match,resolve_params

INSTRUCTION = "pick up the milk from the table"

match = underspecified(PickUpAction)(
    object_designator=...,
    arm=...,
    grasp_description=...,
)

# Role 2 — LLM fills all free slots from world context
plan_node = resolve_match(
    match           = match,
    context         = context,
    llm             = llm,
    groundable_type = groundable_type,
    instruction     = INSTRUCTION,   # optional — helps with entity grounding
)
print("Backend type :", type(context.query_backend).__name__)
print("Plan node    :", plan_node)


In [ ]:
resolved_action = resolve_params(
    match           = match,
    llm             = llm,
    groundable_type = groundable_type,
    instruction     = INSTRUCTION,   # optional — helps with entity grounding
)

In [ ]:
resolved_action

In [ ]:
plan_node.__dict__

In [ ]:
plan_node.perform()

In [ ]:
plan_node.plan.plan_graph.nodes()

---
## 2 — Context-level

Pass `LLMBackend` at context construction. Every plan in this context goes
through the LLM — no per-plan wiring needed.  Update `instruction` per-plan.


In [ ]:
from krrood.entity_query_language.factories import underspecified
from pycram.robot_plans.actions.core.pick_up import PickUpAction
from pycram.robot_plans.actions.core.navigation import NavigateAction
from pycram.datastructures.dataclasses import Context

from llm_reasoner.backend import LLMBackend
from llm_reasoner.pycram_bridge import execute_single

llm_context = Context.from_world(
    world,
    query_backend=LLMBackend(
        llm=llm,
        groundable_type=groundable_type,
        instruction="go to the kitchen table",
    )
)

print("Backend type:", type(llm_context.query_backend).__name__)


In [ ]:
# Plan 1 — uses the backend set on llm_context
nav_match = underspecified(NavigateAction)(target_location=...)
nav_node = execute_single(nav_match, llm_context)
nav_node.perform()

In [ ]:
# Plan 2 — update instruction for the next action, same context
llm_context.query_backend.instruction = "pick up the milk from the table"

pick_match = underspecified(PickUpAction)(object_designator=..., arm=..., grasp_description=...)
pick_node = execute_single(pick_match, llm_context)
pick_node.perform()

---
## 3 — Mid-session swap

Start with `ProbabilisticBackend` for routine actions. Switch to `LLMBackend`
via `resolve_match()` for novel underspecified actions, then restore.


In [ ]:
from krrood.entity_query_language.backends import ProbabilisticBackend
from krrood.entity_query_language.factories import underspecified
from pycram.robot_plans.actions.core import PickUpAction, PlaceAction

from llm_reasoner.pycram_bridge import execute_single
from llm_reasoner.backend import LLMBackend

# Session starts with probabilistic backend
prob_backend = ProbabilisticBackend()
context.query_backend = prob_backend
print("Start  — backend:", type(context.query_backend).__name__)


In [ ]:
# Routine pick — handled by probabilistic backend
pick_match = underspecified(PickUpAction)(object_designator=..., arm=..., grasp_description=...)
pick_node = execute_single(pick_match, context)
pick_node.perform()

In [ ]:
from krrood.entity_query_language.factories import underspecified
from pycram.robot_plans.actions.core import PlaceAction

from llm_reasoner import resolve_match

# Novel action — use resolve_match (Role 2 entry point)
place_match = underspecified(PlaceAction)(object_designator=..., target_location=...)

place_node = resolve_match(
    match           = place_match,
    context         = context,
    llm             = llm,
    groundable_type = groundable_type,
    instruction     = "put the milk gently inside the fridge on the top shelf",
)
print("Swapped — backend:", type(context.query_backend).__name__)
place_node.perform()

# Restore probabilistic backend
context.query_backend = prob_backend
print("Restored — backend:", type(context.query_backend).__name__)


In [ ]:
# Restore probabilistic backend for remaining session
context.query_backend = prob_backend
print("Restored — backend:", type(context.query_backend).__name__)